In [90]:
import pandas as pd
import numpy as np

In [92]:
df = pd.read_csv('data/house_price_prediction.csv')

In [94]:
df.head()

,avg_income,avg_area_house_age,avg_area_num_rooms,avg_bedrooms,avg_population,price,address
0,79545.45857,5.682861,7.009188,4.09,23086.80050,1.059034e+06,"208 Michael Ferry Apt. 674\nLaurabury, NE 3701..."
1,79248.64245,6.002900,6.730821,3.09,40173.07217,1.505891e+06,"188 Johnson Views Suite 079\nLake Kathleen, CA..."
2,61287.06718,5.865890,8.512727,5.13,36882.15940,1.058988e+06,"9127 Elizabeth Stravenue\nDanieltown, WI 06482..."
3,63345.24005,7.188236,5.586729,3.26,34310.24283,1.260617e+06,USS Barnett\nFPO AP 44820
4,59982.19723,5.040555,7.839388,4.23,26354.10947,6.309435e+05,USNS Raymond\nFPO AE 09386


In [96]:
df.shape

(5012, 7)

In [98]:
df.columns

Index(['avg_income', 'avg_area_house_age', 'avg_area_num_rooms',
       'avg_bedrooms', 'avg_population', 'price', 'address'],
      dtype='object')

In [100]:
df.dtypes

avg_income            float64
avg_area_house_age    float64
avg_area_num_rooms    float64
avg_bedrooms          float64
avg_population        float64
price                 float64
address                object
dtype: object

In [102]:
df.describe()

,avg_income,avg_area_house_age,avg_area_num_rooms,avg_bedrooms,avg_population,price
count,5009.000000,5010.000000,5011.000000,5012.000000,5009.000000,5.012000e+03
mean,68577.804938,5.977508,6.987906,3.982095,36155.832559,1.231947e+06
std,10661.033166,0.991472,1.006453,1.233945,9928.823462,3.529525e+05
min,17796.631190,2.644304,3.236194,2.000000,172.610686,1.593866e+04
25%,61482.244790,5.322274,6.298437,3.140000,29403.512060,9.981375e+05
50%,68814.925610,5.969828,7.003188,4.050000,36183.287800,1.232983e+06
75%,75780.621120,6.652302,7.667048,4.490000,42841.741620,1.471756e+06
max,107701.748400,9.519088,10.759588,6.500000,69621.713380,2.469066e+06


In [104]:
import seaborn as sns
import matplotlib.pyplot as plt

In [106]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
import joblib

In [108]:
import re
import pandas as pd

def parse_address(address):
    # Handle NaN or non-string values
    if pd.isna(address):
        return {
            "street": None,
            "apartment": None,
            "city": None,
            "state": None,
            "zipcode": None
        }

    address = str(address).strip()

    result = {
        "street": None,
        "apartment": None,
        "city": None,
        "state": None,
        "zipcode": None
    }

    parts = address.split("\n")

    # Street & apartment
    apt_match = re.search(r"(Apt\.?\s*\d+)", parts[0])
    if apt_match:
        result["apartment"] = apt_match.group(1)
        result["street"] = parts[0].replace(apt_match.group(1), "").strip()
    else:
        result["street"] = parts[0].strip()

    # City, State, Zip
    if len(parts) > 1:
        city_state_zip = parts[1]

        city_match = re.match(r"([^,]+)", city_state_zip)
        if city_match:
            result["city"] = city_match.group(1).strip()

        state_match = re.search(r",\s*([A-Z]{2})\s+", city_state_zip)
        if state_match:
            result["state"] = state_match.group(1)

        zip_match = re.search(r"(\d{5})(?:-\d{4})?", city_state_zip)
        if zip_match:
            result["zipcode"] = zip_match.group(1)

    return result


In [110]:
df_parsed = df["address"].apply(parse_address).apply(pd.Series)

df = pd.concat([df, df_parsed], axis=1)
df.drop(columns=["address"], inplace=True)


In [112]:
df

,avg_income,avg_area_house_age,avg_area_num_rooms,avg_bedrooms,avg_population,price,street,apartment,city,state,zipcode
0,79545.45857,5.682861,7.009188,4.09,23086.80050,1.059034e+06,208 Michael Ferry,Apt. 674,Laurabury,NE,37010
1,79248.64245,6.002900,6.730821,3.09,40173.07217,1.505891e+06,188 Johnson Views Suite 079,None,Lake Kathleen,CA,48958
2,61287.06718,5.865890,8.512727,5.13,36882.15940,1.058988e+06,9127 Elizabeth Stravenue,None,Danieltown,WI,06482
3,63345.24005,7.188236,5.586729,3.26,34310.24283,1.260617e+06,USS Barnett,None,FPO AP 44820,None,44820
4,59982.19723,5.040555,7.839388,4.23,26354.10947,6.309435e+05,USNS Raymond,None,FPO AE 09386,None,09386
...,...,...,...,...,...,...,...,...,...,...,...
5007,60567.94414,7.830362,6.137356,3.46,22837.36103,1.060194e+06,USNS Williams,None,FPO AP 30153-7653,None,30153
5008,NaN,6.999135,6.576763,4.02,25616.11549,1.482618e+06,"PSC 9258, Box 8489",None,APO AA 42991-3352,None,42991
5009,NaN,7.250591,4.805081,2.13,33266.14549,1.030730e+06,4215 Tracy Garden Suite 076,None,Joshualand,VA,01707
5010,68001.33124,5.534388,7.130144,5.44,42625.62016,1.198657e+06,USS Wallace,None,FPO AE 73316,None,73316


In [114]:
y = df['price']

In [116]:
X = df.drop(columns=["price", "street", "apartment"])

In [120]:
X

,avg_income,avg_area_house_age,avg_area_num_rooms,avg_bedrooms,avg_population,city,state,zipcode
0,79545.45857,5.682861,7.009188,4.09,23086.80050,Laurabury,NE,37010
1,79248.64245,6.002900,6.730821,3.09,40173.07217,Lake Kathleen,CA,48958
2,61287.06718,5.865890,8.512727,5.13,36882.15940,Danieltown,WI,06482
3,63345.24005,7.188236,5.586729,3.26,34310.24283,FPO AP 44820,None,44820
4,59982.19723,5.040555,7.839388,4.23,26354.10947,FPO AE 09386,None,09386
...,...,...,...,...,...,...,...,...
5007,60567.94414,7.830362,6.137356,3.46,22837.36103,FPO AP 30153-7653,None,30153
5008,NaN,6.999135,6.576763,4.02,25616.11549,APO AA 42991-3352,None,42991
5009,NaN,7.250591,4.805081,2.13,33266.14549,Joshualand,VA,01707
5010,68001.33124,5.534388,7.130144,5.44,42625.62016,FPO AE 73316,None,73316


In [130]:
X['state'].isna().sum()

522

In [134]:
X['state'] = X.groupby('city')['state'].transform(
    lambda x: x.fillna(x.mode()[0]) if not x.mode().empty else x
)


In [136]:
X['state'].isna().sum()

522